# Phase 10: Temperature-Dependent Device Physics

This notebook characterizes the 4H-SiC detector response across the 280-350K temperature range,
with particular emphasis on the 303-313K clinical range for hadrontherapy dosimetry.

**Sections:**
1. Material Properties vs Temperature (E_g, n_i, mobility, lifetime)
2. I-V Characteristics vs Temperature
3. C-V and Depletion Width vs Temperature
4. CCE vs Temperature
5. Clinical Temperature Coefficient Extraction (303-313K)
6. Summary Table

In [ ]:
import sys
sys.path.insert(0, '..')

%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm

from src.sic_material import (
    SiC4H_Parameters,
    bandgap,
    intrinsic_concentration,
    mobility_caughey_thomas_T,
    srh_lifetime,
)
from src.temperature_sweep import (
    sweep_iv_vs_temperature,
    sweep_cv_vs_temperature,
    sweep_cce_vs_temperature,
    extract_temperature_coefficient,
)

# Publication-quality settings
plt.rcParams.update({
    'figure.dpi': 150,
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'legend.fontsize': 10,
    'figure.figsize': (8, 5),
})

params = SiC4H_Parameters()
print('Notebook 06: Temperature-Dependent Device Physics')
print(f'Temperature range: 280-350 K')
print(f'Clinical range: 303-313 K (30-40 C)')

## Section 1: Material Properties vs Temperature

Fundamental 4H-SiC material properties as a function of temperature:
bandgap (Varshni), intrinsic concentration, electron/hole mobility (Caughey-Thomas),
and SRH lifetimes (power-law scaling).

In [ ]:
T_range = np.linspace(280, 350, 200)

# Compute material properties across T
E_g_arr = np.array([bandgap(T, params) for T in T_range])
n_i_arr = np.array([intrinsic_concentration(T, params)[0] for T in T_range])
mu_n_arr = np.array([mobility_caughey_thomas_T(1e14, T, 'electron', params) for T in T_range])
mu_p_arr = np.array([mobility_caughey_thomas_T(1e14, T, 'hole', params) for T in T_range])
tau_n_arr = np.array([srh_lifetime(T, 'electron', params) for T in T_range])
tau_p_arr = np.array([srh_lifetime(T, 'hole', params) for T in T_range])

# Reference values at 300K
E_g_300 = bandgap(300, params)
n_i_300 = intrinsic_concentration(300, params)[0]
mu_n_300 = mobility_caughey_thomas_T(1e14, 300, 'electron', params)
mu_p_300 = mobility_caughey_thomas_T(1e14, 300, 'hole', params)
tau_n_300 = srh_lifetime(300, 'electron', params)

fig, axes = plt.subplots(2, 3, figsize=(15, 9))

# E_g(T)
ax = axes[0, 0]
ax.plot(T_range, E_g_arr, 'b-', linewidth=2)
ax.axvline(300, color='gray', linestyle='--', alpha=0.5)
ax.annotate(f'{E_g_300:.4f} eV', xy=(300, E_g_300), fontsize=9,
            xytext=(315, E_g_300+0.002), arrowprops=dict(arrowstyle='->', color='red'))
ax.set_xlabel('Temperature (K)')
ax.set_ylabel('$E_g$ (eV)')
ax.set_title('Bandgap (Varshni)')
ax.grid(True, alpha=0.3)

# n_i(T)
ax = axes[0, 1]
ax.semilogy(T_range, n_i_arr, 'r-', linewidth=2)
ax.axvline(300, color='gray', linestyle='--', alpha=0.5)
ax.annotate(f'{n_i_300:.2e}', xy=(300, n_i_300), fontsize=9,
            xytext=(315, n_i_300*5), arrowprops=dict(arrowstyle='->', color='red'))
ax.set_xlabel('Temperature (K)')
ax.set_ylabel('$n_i$ (cm$^{-3}$)')
ax.set_title('Intrinsic Concentration')
ax.grid(True, alpha=0.3)

# mu_n(T)
ax = axes[0, 2]
ax.plot(T_range, mu_n_arr, 'g-', linewidth=2)
ax.axvline(300, color='gray', linestyle='--', alpha=0.5)
ax.annotate(f'{mu_n_300:.0f} cm$^2$/Vs', xy=(300, mu_n_300), fontsize=9,
            xytext=(315, mu_n_300+50), arrowprops=dict(arrowstyle='->', color='red'))
ax.set_xlabel('Temperature (K)')
ax.set_ylabel('$\\mu_n$ (cm$^2$/Vs)')
ax.set_title('Electron Mobility (Caughey-Thomas)')
ax.grid(True, alpha=0.3)

# mu_p(T)
ax = axes[1, 0]
ax.plot(T_range, mu_p_arr, 'm-', linewidth=2)
ax.axvline(300, color='gray', linestyle='--', alpha=0.5)
ax.annotate(f'{mu_p_300:.0f} cm$^2$/Vs', xy=(300, mu_p_300), fontsize=9,
            xytext=(315, mu_p_300+5), arrowprops=dict(arrowstyle='->', color='red'))
ax.set_xlabel('Temperature (K)')
ax.set_ylabel('$\\mu_p$ (cm$^2$/Vs)')
ax.set_title('Hole Mobility (Caughey-Thomas)')
ax.grid(True, alpha=0.3)

# tau_n(T)
ax = axes[1, 1]
ax.plot(T_range, tau_n_arr * 1e9, 'c-', linewidth=2, label='$\\tau_n$')
ax.plot(T_range, tau_p_arr * 1e6, 'orange', linewidth=2, label='$\\tau_p$ (us scale)')
ax.axvline(300, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Temperature (K)')
ax.set_ylabel('Lifetime')
ax.set_title('SRH Lifetimes')
ax.legend()
ax.grid(True, alpha=0.3)
# Add dual y-axis label
ax.text(0.02, 0.95, '$\\tau_n$ in ns, $\\tau_p$ in $\\mu$s', transform=ax.transAxes,
        fontsize=8, va='top')

# Summary text
ax = axes[1, 2]
ax.axis('off')
summary_text = (
    'Material Property Trends\n'
    '(280-350K range)\n\n'
    f'$E_g$: {E_g_arr[-1]-E_g_arr[0]:.4f} eV change\n'
    f'$n_i$: {n_i_arr[-1]/n_i_arr[0]:.1f}x increase\n'
    f'$\\mu_n$: {(1-mu_n_arr[-1]/mu_n_arr[0])*100:.0f}% decrease\n'
    f'$\\mu_p$: {(1-mu_p_arr[-1]/mu_p_arr[0])*100:.0f}% decrease\n'
    f'$\\tau_n$: {tau_n_arr[-1]/tau_n_arr[0]:.2f}x increase\n'
)
ax.text(0.1, 0.9, summary_text, transform=ax.transAxes,
        fontsize=11, va='top', family='monospace',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.show()

## Section 2: I-V Characteristics vs Temperature

Reverse leakage current increases strongly with temperature due to the exponential
dependence of $n_i$ on $T$. This section sweeps 8 temperatures from 280-350K.

In [ ]:
# I-V sweep across 8 temperatures
T_iv = [280, 290, 300, 310, 320, 330, 340, 350]
df_iv = sweep_iv_vs_temperature(T_iv, V_reverse=-30)

print('I-V Temperature Sweep Results:')
print(df_iv.to_string(index=False))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: I_reverse vs T
ax1.semilogy(df_iv['T'], df_iv['I_reverse'], 'ro-', markersize=6, linewidth=2)
ax1.set_xlabel('Temperature (K)')
ax1.set_ylabel('$|I_{reverse}|$ at $V=-30$V (A/cm$^2$)')
ax1.set_title('Reverse Leakage Current vs Temperature')
ax1.grid(True, alpha=0.3)

# Right: n_i vs T for context
ax2.semilogy(df_iv['T'], df_iv['n_i'], 'bs-', markersize=6, linewidth=2)
ax2.set_xlabel('Temperature (K)')
ax2.set_ylabel('$n_i$ (cm$^{-3}$)')
ax2.set_title('Intrinsic Concentration vs Temperature')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Compute I_reverse ratio
I_280 = df_iv[df_iv['T']==280]['I_reverse'].values[0]
I_350 = df_iv[df_iv['T']==350]['I_reverse'].values[0]
print(f'\nI_reverse ratio (350K/280K): {I_350/I_280:.1f}x')
print(f'n_i ratio (350K/280K): {df_iv[df_iv["T"]==350]["n_i"].values[0]/df_iv[df_iv["T"]==280]["n_i"].values[0]:.1f}x')

## Section 3: C-V and Depletion Width vs Temperature

The built-in potential $V_{bi} = (kT/q) \ln(N_A \cdot N_D / n_i^2)$ shifts slightly
with temperature. For 4H-SiC, this effect is small because of the large bandgap.

In [ ]:
# C-V sweep at 4 temperatures
T_cv = [280, 300, 320, 350]
voltages_cv = np.array([0, -5, -10, -15, -20, -25, -30], dtype=float)
cv_result = sweep_cv_vs_temperature(T_cv, voltages=voltages_cv)

print('Depletion width at 0V by temperature:')
for i, T in enumerate(T_cv):
    print(f'  T={T}K: W(0V) = {cv_result["W"][i,0]*1e4:.3f} um')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

colors_cv = ['blue', 'green', 'orange', 'red']
for i, T in enumerate(T_cv):
    W_um = cv_result['W'][i, :] * 1e4  # convert to um
    ax1.plot(voltages_cv, W_um, 'o-', color=colors_cv[i], 
             label=f'T={T}K', markersize=5, linewidth=1.5)

ax1.set_xlabel('Applied Bias (V)')
ax1.set_ylabel('Depletion Width ($\\mu$m)')
ax1.set_title('$W(V)$ at Different Temperatures')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Capacitance plot
for i, T in enumerate(T_cv):
    C_pF = cv_result['C'][i, :] * 1e12  # F/cm^2 -> pF/cm^2
    ax2.plot(voltages_cv, C_pF, 'o-', color=colors_cv[i],
             label=f'T={T}K', markersize=5, linewidth=1.5)

ax2.set_xlabel('Applied Bias (V)')
ax2.set_ylabel('Capacitance (pF/cm$^2$)')
ax2.set_title('$C(V)$ at Different Temperatures')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Section 4: CCE vs Temperature

Charge collection efficiency depends on temperature through mobility (drift length)
and lifetime (trapping). At high bias (fully depleted), mobility degradation with T
slightly reduces CCE.

In [ ]:
# CCE vs voltage at 4 temperatures
T_cce = [280, 300, 320, 350]
V_cce = np.array([-5, -10, -15, -20, -25, -30], dtype=float)
df_cce = sweep_cce_vs_temperature(T_cce, voltages=V_cce, method='hecht')

print('CCE at V=-30V by temperature:')
for T in T_cce:
    cce_val = df_cce[(df_cce['T']==T) & (df_cce['V']==-30)]['CCE'].values[0]
    print(f'  T={T}K: CCE = {cce_val:.4f}')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: CCE vs voltage at different T
colors_cce = ['blue', 'green', 'orange', 'red']
for i, T in enumerate(T_cce):
    mask = df_cce['T'] == T
    ax1.plot(df_cce[mask]['V'], df_cce[mask]['CCE'], 'o-', color=colors_cce[i],
             label=f'T={T}K', markersize=5, linewidth=1.5)

ax1.set_xlabel('Applied Bias (V)')
ax1.set_ylabel('CCE')
ax1.set_title('CCE vs Voltage at Different Temperatures')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_ylim([0, 1.05])

# Right: CCE at V=-30V vs T
cce_at_30V = df_cce[df_cce['V'] == -30.0].sort_values('T')
ax2.plot(cce_at_30V['T'], cce_at_30V['CCE'], 'rs-', markersize=8, linewidth=2)
ax2.set_xlabel('Temperature (K)')
ax2.set_ylabel('CCE at $V=-30$V')
ax2.set_title('CCE Temperature Dependence (V=-30V)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Section 5: Clinical Temperature Coefficient (303-313K)

Extract the temperature coefficient $dCCE/dT$ and $dI/dT$ over the clinical
temperature range (30-40 degrees C). This is the key metric for dosimetry stability.

In [ ]:
# Clinical range sweep: 303-313K (11 points)
T_clinical = np.linspace(303, 313, 11)

# CCE coefficient
df_clinical_cce = sweep_cce_vs_temperature(
    T_clinical, voltages=np.array([-30.0]), method='hecht'
)
cce_values = df_clinical_cce['CCE'].values

cce_coeff = extract_temperature_coefficient(T_clinical, cce_values, 'CCE')

print('=== Clinical Temperature Coefficient: CCE ===')
print(f'  dCCE/dT = {cce_coeff["slope"]:.6f} /K')
print(f'  dCCE/dT = {cce_coeff["slope"]*100:.4f} %/K')
print(f'  R^2     = {cce_coeff["r_squared"]:.6f}')
print(f'  CCE range: {cce_values.min():.4f} - {cce_values.max():.4f}')

In [ ]:
# Leakage current coefficient
df_clinical_iv = sweep_iv_vs_temperature(T_clinical, V_reverse=-30)
I_values = df_clinical_iv['I_reverse'].values

I_coeff = extract_temperature_coefficient(T_clinical, I_values, 'I_reverse')

print('=== Clinical Temperature Coefficient: Leakage Current ===')
print(f'  dI/dT   = {I_coeff["slope"]:.4e} A/cm^2/K')
print(f'  R^2     = {I_coeff["r_squared"]:.6f}')
print(f'  I range: {I_values.min():.4e} - {I_values.max():.4e} A/cm^2')

In [ ]:
# Publication-quality figure: CCE vs T with linear fit
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: CCE vs T with fit
ax1.plot(T_clinical, cce_values, 'ro', markersize=6, label='Hecht CCE')
T_fit = np.linspace(T_clinical[0]-1, T_clinical[-1]+1, 100)
cce_fit = cce_coeff['slope'] * T_fit + cce_coeff['intercept']
ax1.plot(T_fit, cce_fit, 'b-', linewidth=1.5, label='Linear fit')
ax1.set_xlabel('Temperature (K)')
ax1.set_ylabel('CCE at $V=-30$V')
ax1.set_title('CCE Temperature Coefficient (Clinical Range)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Annotate with fit equation
eq_text = f'dCCE/dT = {cce_coeff["slope"]*100:.4f} %/K\n$R^2$ = {cce_coeff["r_squared"]:.4f}'
ax1.text(0.05, 0.15, eq_text, transform=ax1.transAxes, fontsize=10,
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

# Right: I_reverse vs T with fit
ax2.plot(T_clinical, I_values, 'ro', markersize=6, label='DD leakage')
I_fit = I_coeff['slope'] * T_fit + I_coeff['intercept']
ax2.plot(T_fit, I_fit, 'b-', linewidth=1.5, label='Linear fit')
ax2.set_xlabel('Temperature (K)')
ax2.set_ylabel('$|I_{reverse}|$ at $V=-30$V (A/cm$^2$)')
ax2.set_title('Leakage Current Temperature Coefficient (Clinical Range)')
ax2.legend()
ax2.grid(True, alpha=0.3)

eq_text2 = f'dI/dT = {I_coeff["slope"]:.4e} A/cm$^2$/K\n$R^2$ = {I_coeff["r_squared"]:.4f}'
ax2.text(0.05, 0.85, eq_text2, transform=ax2.transAxes, fontsize=10,
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.show()

## Section 6: Summary Table

Key temperature coefficients and sensitivity metrics for the 4H-SiC detector.

In [ ]:
# Compute summary metrics
E_g_280 = bandgap(280, params)
E_g_350 = bandgap(350, params)
dEg_dT = (E_g_350 - E_g_280) / (350 - 280)

n_i_280 = intrinsic_concentration(280, params)[0]
n_i_350 = intrinsic_concentration(350, params)[0]

mu_n_280 = mobility_caughey_thomas_T(1e14, 280, 'electron', params)
mu_n_350 = mobility_caughey_thomas_T(1e14, 350, 'electron', params)
dmu_dT = (mu_n_350 - mu_n_280) / (350 - 280)

print('=' * 70)
print('TEMPERATURE SENSITIVITY SUMMARY')
print('4H-SiC Detector at V=-30V Reverse Bias')
print('=' * 70)
print()
print(f'{"Property":<35s} {"Value":<25s} {"Range":>10s}')
print('-' * 70)
print(f'{"dE_g/dT (Varshni)":<35s} {dEg_dT*1000:.3f} meV/K{"":>12s}')
print(f'{"n_i ratio (350K/280K)":<35s} {n_i_350/n_i_280:.1f}x{"":>15s}')
print(f'{"dmu_n/dT":<35s} {dmu_dT:.2f} cm^2/Vs/K{"":>8s}')
print(f'{"dCCE/dT (clinical, 303-313K)":<35s} {cce_coeff["slope"]*100:.4f} %/K{"":>11s}')
print(f'{"dI_rev/dT (clinical, 303-313K)":<35s} {I_coeff["slope"]:.4e} A/cm^2/K{"":>2s}')
print(f'{"CCE R^2 (linearity)":<35s} {cce_coeff["r_squared"]:.6f}{"":>14s}')
print()
print('Key takeaways:')
print('  - Wide-bandgap SiC is inherently temperature-stable compared to Si')
print('  - n_i remains negligible (<< N_D) across the full range')
print('  - CCE variation in the clinical range is very small')
print('  - Leakage current increases with T but remains extremely low')